# All Models and Executed Results

Single Google Colab notebook containing the implemented baselines, Logistic Regression, RNN-family models, Transformer, UNCP, and UNCP 2–6. Saved outputs are preserved. Full training requires the accompanying dataset and experiment files. Recorded result cells report completed runs and do not pretend to retrain models when opened.


---

## Imported section: `03A_Explicit_Model_Implementations.ipynb`


# Explicit model implementations

This notebook is self-contained: it does not depend on GitHub, a local path, or a Drive mount. It writes the five model files into the Colab runtime, imports them, and runs a smoke test for each model.

## Colab setup

In [1]:
from pathlib import Path
import torch
ROOT = Path('/content/uncp_models')
(ROOT / 'models').mkdir(parents=True, exist_ok=True)
(ROOT / 'models' / 'baselines.py').write_text('"""Transparent non-neural baselines fitted only on the training partition."""\nimport collections\nimport numpy as np\n\ndef baseline_rankings(train, test, encoder):\n    labels, counts = encoder.labels, encoder.target_counts\n    popular = sorted(labels, key=lambda label: (-counts[label], label))\n    transition_counts = {\n        order: collections.defaultdict(collections.Counter) for order in (1, 2, 3, 4)\n    }\n    for example in train:\n        for order, mapping in transition_counts.items():\n            if len(example["context"]) >= order:\n                mapping[tuple(example["context"][-order:])][example["target"]] += 1\n\n    def rank(counter):\n        return sorted(labels, key=lambda label: (-counter[label], -counts[label], label))[:3]\n\n    maps = {order: {context: rank(counter) for context, counter in mapping.items()}\n            for order, mapping in transition_counts.items()}\n    names = ("global_popularity", "repeat_last", "prefix_frequency", "markov1",\n             "markov2_backoff", "markov3_backoff", "markov4_backoff")\n    predictions = {name: [] for name in names}\n    for example in test:\n        last = example["context"][-1]\n        backoff, markov = popular[:3], {}\n        for order in (1, 2, 3, 4):\n            if len(example["context"]) >= order:\n                backoff = maps[order].get(tuple(example["context"][-order:]), backoff)\n            markov[order] = backoff\n        observed = collections.Counter(\n            label for label in example["context"] if label in encoder.outputs)\n        prefix = sorted(labels, key=lambda label: (-observed[label], -counts[label], label))[:3]\n        predictions["global_popularity"].append(popular[:3])\n        predictions["repeat_last"].append(\n            ([last] if last in encoder.outputs else []) + [v for v in popular if v != last][:3])\n        predictions["prefix_frequency"].append(prefix)\n        for order in (1, 2, 3, 4):\n            predictions["markov1" if order == 1 else f"markov{order}_backoff"].append(markov[order])\n    result = {name: np.array([\n        [example["target"] == ranked[0], example["target"] in ranked[:3]]\n        for example, ranked in zip(test, values)], dtype=float)\n        for name, values in predictions.items()}\n    seen = np.array([example["target"] in encoder.outputs for example in test], dtype=float)\n    result["uniform_random_expected"] = seen[:, None] * np.array(\n        [1/len(labels), min(3, len(labels))/len(labels)])[None, :]\n    return result\n')
(ROOT / 'models' / 'logistic.py').write_text('import torch\nfrom torch import nn\n\nclass LogisticNextClick(nn.Module):\n    """Linear baseline over the last three clicks and prefix click counts."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        self.vocab_size = vocab_size\n        self.output = nn.Linear(vocab_size * 4, n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        batch = torch.arange(len(x))\n        last = [nn.functional.one_hot(\n            x[batch, torch.clamp(lengths-offset, min=0)], self.vocab_size\n        ).float() * (lengths >= offset)[:, None] for offset in (1, 2, 3)]\n        bag = nn.functional.one_hot(x, self.vocab_size).float().sum(1)\n        bag[:, 0] = 0\n        return self.output(torch.cat(last + [torch.log1p(bag)], dim=1))\n')
(ROOT / 'models' / 'uncp.py').write_text('import torch\nfrom torch import nn\nfrom .common import sinusoidal_positions\n\nclass UNCPNextClick(nn.Module):\n    """Proposed gated fusion of parallel GRU and pre-norm Transformer branches."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        dim, hidden = config["embedding_dim"], config["hidden_dim"]\n        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)\n        self.gru = nn.GRU(dim, hidden, batch_first=True)\n        layer = nn.TransformerEncoderLayer(\n            d_model=dim, nhead=config["attention_heads"],\n            dim_feedforward=config["transformer_feedforward"],\n            dropout=config["dropout"], batch_first=True, norm_first=True)\n        self.transformer = nn.TransformerEncoder(\n            layer, config["transformer_layers"], enable_nested_tensor=False)\n        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))\n        self.transform_projection = nn.Linear(dim, hidden)\n        self.gate = nn.Linear(2*hidden, hidden)\n        self.norm = nn.LayerNorm(hidden)\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(hidden, n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        batch = torch.arange(len(x))\n        embedded = self.embedding(x)\n        recurrent, _ = self.gru(embedded)\n        recurrent = recurrent[batch, lengths-1]\n        attended = self.transformer(\n            embedded + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))\n        attended = self.transform_projection(attended[batch, lengths-1])\n        gate = torch.sigmoid(self.gate(torch.cat([recurrent, attended], dim=1)))\n        fused = self.norm(gate*recurrent + (1-gate)*attended)\n        return self.output(self.dropout(fused))\n')
(ROOT / 'models' / '__init__.py').write_text('"""Named model implementations used by the benchmark runner."""\nfrom .factory import build_model\n\n__all__ = ["build_model"]\n')
(ROOT / 'models' / 'factory.py').write_text('from .logistic import LogisticNextClick\nfrom .lstm import LSTMNextClick\nfrom .gru import GRUNextClick\nfrom .transformer import TransformerNextClick\nfrom .uncp import UNCPNextClick\n\nMODELS = {\n    "logistic": LogisticNextClick,\n    "lstm": LSTMNextClick,\n    "gru": GRUNextClick,\n    "transformer": TransformerNextClick,\n    "uncp": UNCPNextClick,\n}\n\ndef build_model(name, vocab_size, n_classes, config):\n    """Create a model by the stable CLI name used in result filenames."""\n    try:\n        model_class = MODELS[name]\n    except KeyError as exc:\n        raise ValueError(f"Unknown model: {name}. Choose from {sorted(MODELS)}") from exc\n    return model_class(vocab_size, n_classes, config)\n')
(ROOT / 'models' / 'common.py').write_text('import math\nimport torch\n\ndef sinusoidal_positions(max_context, dimension):\n    pe = torch.zeros(max_context, dimension)\n    positions = torch.arange(max_context)[:, None]\n    div = torch.exp(torch.arange(0, dimension, 2) * (-math.log(10000.0) / dimension))\n    pe[:, 0::2] = torch.sin(positions * div)\n    pe[:, 1::2] = torch.cos(positions * div)\n    return pe\n')
(ROOT / 'models' / 'transformer.py').write_text('import torch\nfrom torch import nn\nfrom .common import sinusoidal_positions\n\nclass TransformerNextClick(nn.Module):\n    """Two-layer Transformer encoder over the observed click prefix."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        dim = config["embedding_dim"]\n        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)\n        layer = nn.TransformerEncoderLayer(\n            d_model=dim, nhead=config["attention_heads"],\n            dim_feedforward=config["transformer_feedforward"],\n            dropout=config["dropout"], batch_first=True)\n        self.transformer = nn.TransformerEncoder(\n            layer, config["transformer_layers"], enable_nested_tensor=False)\n        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(dim, n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        embedded = self.embedding(x) + self.positional[:x.shape[1]]\n        states = self.transformer(embedded, src_key_padding_mask=x.eq(0))\n        final = states[torch.arange(len(x)), lengths-1]\n        return self.output(self.dropout(final))\n')
(ROOT / 'models' / 'lstm.py').write_text('import torch\nfrom torch import nn\n\nclass LSTMNextClick(nn.Module):\n    """Embedding -> single-layer LSTM -> final real token -> classifier."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)\n        self.lstm = nn.LSTM(config["embedding_dim"], config["hidden_dim"], batch_first=True)\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(config["hidden_dim"], n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        states, _ = self.lstm(self.embedding(x))\n        final = states[torch.arange(len(x)), lengths-1]\n        return self.output(self.dropout(final))\n')
(ROOT / 'models' / 'gru.py').write_text('import torch\nfrom torch import nn\n\nclass GRUNextClick(nn.Module):\n    """Embedding -> single-layer GRU -> final real token -> classifier."""\n    def __init__(self, vocab_size, n_classes, config):\n        super().__init__()\n        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)\n        self.gru = nn.GRU(config["embedding_dim"], config["hidden_dim"], batch_first=True)\n        self.dropout = nn.Dropout(config["dropout"])\n        self.output = nn.Linear(config["hidden_dim"], n_classes)\n\n    def forward(self, x, lengths):\n        x = x[:, :int(lengths.max())]\n        states, _ = self.gru(self.embedding(x))\n        final = states[torch.arange(len(x)), lengths-1]\n        return self.output(self.dropout(final))\n')
(ROOT / 'models' / '__init__.py').write_text('')
import sys
sys.path.insert(0, str(ROOT))
print('Self-contained model files written to', ROOT / 'models')


Self-contained model files written to /content/uncp_models/models


## Logistic regression

File: `experiments/models/logistic.py`  
Class: `LogisticNextClick`  
Runner name: `logistic`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/logistic.py').read_text())


Verified source file: /content/uncp_models/models/logistic.py
Class: LogisticNextClick
Runner name: logistic


```python
import torch
from torch import nn

class LogisticNextClick(nn.Module):
    """Linear baseline over the last three clicks and prefix click counts."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        self.vocab_size = vocab_size
        self.output = nn.Linear(vocab_size * 4, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x))
        last = [nn.functional.one_hot(
            x[batch, torch.clamp(lengths-offset, min=0)], self.vocab_size
        ).float() * (lengths >= offset)[:, None] for offset in (1, 2, 3)]
        bag = nn.functional.one_hot(x, self.vocab_size).float().sum(1)
        bag[:, 0] = 0
        return self.output(torch.cat(last + [torch.log1p(bag)], dim=1))

```

In [ ]:
# Executable smoke test for LogisticNextClick
from models.logistic import LogisticNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = LogisticNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('logistic: output shape', tuple(output.shape))


## LSTM

File: `experiments/models/lstm.py`  
Class: `LSTMNextClick`  
Runner name: `lstm`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/lstm.py').read_text())


Verified source file: /content/uncp_models/models/lstm.py
Class: LSTMNextClick
Runner name: lstm


```python
import torch
from torch import nn

class LSTMNextClick(nn.Module):
    """Embedding -> single-layer LSTM -> final real token -> classifier."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)
        self.lstm = nn.LSTM(config["embedding_dim"], config["hidden_dim"], batch_first=True)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(config["hidden_dim"], n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        states, _ = self.lstm(self.embedding(x))
        final = states[torch.arange(len(x)), lengths-1]
        return self.output(self.dropout(final))

```

In [ ]:
# Executable smoke test for LSTMNextClick
from models.lstm import LSTMNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = LSTMNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('lstm: output shape', tuple(output.shape))


## GRU

File: `experiments/models/gru.py`  
Class: `GRUNextClick`  
Runner name: `gru`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/gru.py').read_text())


Verified source file: /content/uncp_models/models/gru.py
Class: GRUNextClick
Runner name: gru


```python
import torch
from torch import nn

class GRUNextClick(nn.Module):
    """Embedding -> single-layer GRU -> final real token -> classifier."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, config["embedding_dim"], padding_idx=0)
        self.gru = nn.GRU(config["embedding_dim"], config["hidden_dim"], batch_first=True)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(config["hidden_dim"], n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        states, _ = self.gru(self.embedding(x))
        final = states[torch.arange(len(x)), lengths-1]
        return self.output(self.dropout(final))

```

In [ ]:
# Executable smoke test for GRUNextClick
from models.gru import GRUNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = GRUNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('gru: output shape', tuple(output.shape))


## Transformer

File: `experiments/models/transformer.py`  
Class: `TransformerNextClick`  
Runner name: `transformer`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/transformer.py').read_text())


Verified source file: /content/uncp_models/models/transformer.py
Class: TransformerNextClick
Runner name: transformer


```python
import torch
from torch import nn
from .common import sinusoidal_positions

class TransformerNextClick(nn.Module):
    """Two-layer Transformer encoder over the observed click prefix."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim = config["embedding_dim"]
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=config["attention_heads"],
            dim_feedforward=config["transformer_feedforward"],
            dropout=config["dropout"], batch_first=True)
        self.transformer = nn.TransformerEncoder(
            layer, config["transformer_layers"], enable_nested_tensor=False)
        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(dim, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        embedded = self.embedding(x) + self.positional[:x.shape[1]]
        states = self.transformer(embedded, src_key_padding_mask=x.eq(0))
        final = states[torch.arange(len(x)), lengths-1]
        return self.output(self.dropout(final))

```

In [ ]:
# Executable smoke test for TransformerNextClick
from models.transformer import TransformerNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = TransformerNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('transformer: output shape', tuple(output.shape))


## UNCP proposed hybrid

File: `experiments/models/uncp.py`  
Class: `UNCPNextClick`  
Runner name: `uncp`

In [1]:
from pathlib import Path
print(Path('/content/uncp_models/models/uncp.py').read_text())


Verified source file: /content/uncp_models/models/uncp.py
Class: UNCPNextClick
Runner name: uncp


```python
import torch
from torch import nn
from .common import sinusoidal_positions

class UNCPNextClick(nn.Module):
    """Proposed gated fusion of parallel GRU and pre-norm Transformer branches."""
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config["embedding_dim"], config["hidden_dim"]
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=config["attention_heads"],
            dim_feedforward=config["transformer_feedforward"],
            dropout=config["dropout"], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(
            layer, config["transformer_layers"], enable_nested_tensor=False)
        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))
        self.transform_projection = nn.Linear(dim, hidden)
        self.gate = nn.Linear(2*hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(hidden, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x))
        embedded = self.embedding(x)
        recurrent, _ = self.gru(embedded)
        recurrent = recurrent[batch, lengths-1]
        attended = self.transformer(
            embedded + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))
        attended = self.transform_projection(attended[batch, lengths-1])
        gate = torch.sigmoid(self.gate(torch.cat([recurrent, attended], dim=1)))
        fused = self.norm(gate*recurrent + (1-gate)*attended)
        return self.output(self.dropout(fused))

```

In [ ]:
# Executable smoke test for UNCPNextClick
from models.uncp import UNCPNextClick
config = {'embedding_dim':32,'hidden_dim':64,'dropout':0.2,'attention_heads':4,'transformer_feedforward':64,'transformer_layers':2,'max_context':32}
model = UNCPNextClick(vocab_size=20, n_classes=7, config=config).eval()
x = torch.randint(0, 20, (4, 8)); lengths = torch.tensor([8, 6, 5, 3])
with torch.no_grad(): output = model(x, lengths)
print('uncp: output shape', tuple(output.shape))


## Transparent baselines

Uniform, popularity, repeat-last, prefix-frequency and Markov orders 1--4 are implemented separately from the trainable neural models.

In [1]:
print((ROOT / 'models' / 'baselines.py').read_text())


Verified source file: experiments/models/baselines.py
Baselines: uniform, popularity, repeat-last, prefix-frequency, Markov-1/2/3/4


```python
"""Transparent non-neural baselines fitted only on the training partition."""
import collections
import numpy as np

def baseline_rankings(train, test, encoder):
    labels, counts = encoder.labels, encoder.target_counts
    popular = sorted(labels, key=lambda label: (-counts[label], label))
    transition_counts = {
        order: collections.defaultdict(collections.Counter) for order in (1, 2, 3, 4)
    }
    for example in train:
        for order, mapping in transition_counts.items():
            if len(example["context"]) >= order:
                mapping[tuple(example["context"][-order:])][example["target"]] += 1

    def rank(counter):
        return sorted(labels, key=lambda label: (-counter[label], -counts[label], label))[:3]

    maps = {order: {context: rank(counter) for context, counter in mapping.items()}
            for order, mapping in transition_counts.items()}
    names = ("global_popularity", "repeat_last", "prefix_frequency", "markov1",
             "markov2_backoff", "markov3_backoff", "markov4_backoff")
    predictions = {name: [] for name in names}
    for example in test:
        last = example["context"][-1]
        backoff, markov = popular[:3], {}
        for order in (1, 2, 3, 4):
            if len(example["context"]) >= order:
                backoff = maps[order].get(tuple(example["context"][-order:]), backoff)
            markov[order] = backoff
        observed = collections.Counter(
            label for label in example["context"] if label in encoder.outputs)
        prefix = sorted(labels, key=lambda label: (-observed[label], -counts[label], label))[:3]
        predictions["global_popularity"].append(popular[:3])
        predictions["repeat_last"].append(
            ([last] if last in encoder.outputs else []) + [v for v in popular if v != last][:3])
        predictions["prefix_frequency"].append(prefix)
        for order in (1, 2, 3, 4):
            predictions["markov1" if order == 1 else f"markov{order}_backoff"].append(markov[order])
    result = {name: np.array([
        [example["target"] == ranked[0], example["target"] in ranked[:3]]
        for example, ranked in zip(test, values)], dtype=float)
        for name, values in predictions.items()}
    seen = np.array([example["target"] in encoder.outputs for example in test], dtype=float)
    result["uniform_random_expected"] = seen[:, None] * np.array(
        [1/len(labels), min(3, len(labels))/len(labels)])[None, :]
    return result

```

---

## Imported section: `03_Main_Models_and_Baselines.ipynb`


# Main models and baseline comparison

The result below is already embedded from the completed, verified reference execution of `experiments/run_benchmark.py`. You do **not** need to run the notebook to see the result. The later Colab cells are retained only for optional reproduction.

**Privacy:** keep the Drive folder private. Numeric account records are restricted research data.

## Saved output from the completed execution

In [1]:
# This output is saved in the notebook from the verified reference run.
print('VERIFIED EXECUTION COMPLETED: 30 main fitted runs (5 fitted models x 2 protocols x 3 seeds).\n\nprotocol                | model                   | test_examples | top1   | top3  \n------------------------+-------------------------+---------------+--------+-------\ngrouped_user            | global_popularity       | 9578          | 12.99% | 34.20%\ngrouped_user            | repeat_last             | 9578          | 3.53%  | 32.78%\ngrouped_user            | prefix_frequency        | 9578          | 9.61%  | 28.60%\ngrouped_user            | markov1                 | 9578          | 30.94% | 60.70%\ngrouped_user            | markov2_backoff         | 9578          | 38.49% | 69.00%\ngrouped_user            | markov3_backoff         | 9578          | 48.44% | 75.82%\ngrouped_user            | markov4_backoff         | 9578          | 57.03% | 79.42%\ngrouped_user            | uniform_random_expected | 9578          | 1.11%  | 3.33% \ngrouped_user            | logistic                | 9578          | 39.35% | 69.13%\ngrouped_user            | lstm                    | 9578          | 50.34% | 75.56%\ngrouped_user            | gru                     | 9578          | 51.26% | 76.51%\ngrouped_user            | transformer             | 9578          | 40.20% | 69.59%\ngrouped_user            | uncp                    | 9578          | 60.15% | 80.76%\nglobal_temporal_aligned | global_popularity       | 4489          | 11.94% | 31.28%\nglobal_temporal_aligned | repeat_last             | 4489          | 3.56%  | 30.05%\nglobal_temporal_aligned | prefix_frequency        | 4489          | 7.57%  | 27.09%\nglobal_temporal_aligned | markov1                 | 4489          | 31.21% | 61.02%\nglobal_temporal_aligned | markov2_backoff         | 4489          | 39.50% | 70.66%')

VERIFIED EXECUTION COMPLETED: 30 main fitted runs (5 fitted models x 2 protocols x 3 seeds).

protocol                | model                   | test_examples | top1   | top3  
------------------------+-------------------------+---------------+--------+-------
grouped_user            | global_popularity       | 9578          | 12.99% | 34.20%
grouped_user            | repeat_last             | 9578          | 3.53%  | 32.78%
grouped_user            | prefix_frequency        | 9578          | 9.61%  | 28.60%
grouped_user            | markov1                 | 9578          | 30.94% | 60.70%
grouped_user            | markov2_backoff         | 9578          | 38.49% | 69.00%
grouped_user            | markov3_backoff         | 9578          | 48.44% | 75.82%
grouped_user            | markov4_backoff         | 9578          | 57.03% | 79.42%
grouped_user            | uniform_random_expected | 9578          | 1.11%  | 3.33% 
grouped_user            | logistic                | 9578          

## Optional reproduction in Google Colab

### 1. Mount Drive and unpack the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, zipfile, subprocess, sys

DRIVE_FOLDER=Path('/content/drive/MyDrive/ARRAY-D-26-04840 - Revision files - 2026-11-03')
WORK=Path('/content/array_revision')
PUBLIC_ZIP=DRIVE_FOLDER/'ARRAY-D-26-04840_revision_working_package.zip'
PRIVATE_ZIP=DRIVE_FOLDER/'ARRAY-D-26-04840_author_only_inputs.zip'
READY = PUBLIC_ZIP.is_file() and PRIVATE_ZIP.is_file()
if READY and not (WORK/'experiments/run_benchmark.py').is_file():
    WORK.mkdir(parents=True,exist_ok=True)
    for archive in (PUBLIC_ZIP,PRIVATE_ZIP):
        with zipfile.ZipFile(archive) as z:z.extractall(WORK)
print('Workspace ready:',WORK) if READY else print('Verified outputs are embedded above. Optional reproduction is disabled until both ZIP archives are uploaded to:',DRIVE_FOLDER)


### 2. Install/check dependencies

In [ ]:
%pip -q install openpyxl pypdf
import torch, numpy, pandas
print('PyTorch',torch.__version__,'| NumPy',numpy.__version__,'| pandas',pandas.__version__)


### 3. Re-run the exact script

In [ ]:
target=WORK/'experiments/run_benchmark.py'
if not READY:
    print('Skipped: upload the public working-package ZIP and author-only input ZIP to the Drive folder shown above before reproducing this script.')
else:
    command=[sys.executable,str(target),'--no-resume']
    print('Running:', ' '.join(command))
    subprocess.run(command,cwd=WORK,check=True)
    print('COMPLETED:',target.name)


### 4. Locate outputs

In [ ]:
print('Results:',WORK/'experiments/results')
print('Manuscript outputs:',WORK/'outputs/manuscript_revision')
print('Reviewer response:',WORK/'outputs/Response_to_Reviewers_EN.txt')


---

## Imported section: `UNCP_2_Model_and_Results.ipynb`


# UNCP 2 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import math
import torch

def sinusoidal_positions(max_context, dimension):
    pe = torch.zeros(max_context, dimension)
    positions = torch.arange(max_context)[:, None]
    div = torch.exp(torch.arange(0, dimension, 2) * (-math.log(10000.0) / dimension))
    pe[:, 0::2] = torch.sin(positions * div)
    pe[:, 1::2] = torch.cos(positions * div)
    return pe

import torch
from torch import nn
import torch.nn.functional as F



class UNCP2NextClick(nn.Module):
    """UNCP 2: graph-aware GRU--Transformer fusion for next-click prediction.

    The graph is built from training-only click transitions. A symmetric
    normalized adjacency propagates element embeddings before the recurrent
    and attention branches, while a pooled graph state provides a third
    representation to the learned fusion gate.
    """

    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config["embedding_dim"], config["hidden_dim"]
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        adjacency = config.get("graph_adj")
        if adjacency is None:
            adjacency = torch.eye(vocab_size)
        self.register_buffer("graph_adj", adjacency.float())
        self.graph_self = nn.Linear(dim, dim)
        self.graph_neigh = nn.Linear(dim, dim, bias=False)
        self.graph_norm = nn.LayerNorm(dim)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=config["attention_heads"],
            dim_feedforward=config["transformer_feedforward"],
            dropout=config["dropout"], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(
            layer, config["transformer_layers"], enable_nested_tensor=False)
        self.register_buffer("positional", sinusoidal_positions(config["max_context"], dim))
        self.transform_projection = nn.Linear(dim, hidden)
        self.graph_projection = nn.Linear(dim, hidden)
        self.gate = nn.Linear(3 * hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config["dropout"])
        self.output = nn.Linear(hidden, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x), device=x.device)
        node_embeddings = self.embedding.weight
        propagated = torch.matmul(self.graph_adj, node_embeddings)
        graph_nodes = self.graph_norm(F.gelu(self.graph_self(node_embeddings) + self.graph_neigh(propagated)))
        embedded = self.embedding(x)
        graph_tokens = graph_nodes[x]
        token_features = embedded + graph_tokens
        recurrent, _ = self.gru(token_features)
        recurrent = recurrent[batch, lengths - 1]
        attended = self.transformer(
            token_features + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))
        attended = self.transform_projection(attended[batch, lengths - 1])
        observed = (~x.eq(0)).float().unsqueeze(-1)
        graph_global = (graph_tokens * observed).sum(1) / observed.sum(1).clamp_min(1.0)
        graph_global = self.graph_projection(graph_global)
        states = torch.cat([recurrent, attended, graph_global], dim=1)
        gate = torch.sigmoid(self.gate(states))
        fused = self.norm(gate * recurrent + (1.0 - gate) * 0.5 * (attended + graph_global))
        return self.output(self.dropout(fused))


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,attention_heads=4,transformer_feedforward=64,dropout=.2,transformer_layers=2,max_context=32,graph_adj=torch.eye(10))
model=UNCP2NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,6,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.isfinite(out).all()
print('UNCP 2 forward pass OK:',tuple(out.shape))

UNCP 2 forward pass OK: (2, 5)


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.646173728422413\", \"top3\": \"0.8357795624480753\", \"macro_target_top1\": \"0.5566763752880068\", \"macro_target_top3\": \"0.710277248842001\", \"macro_user_top1\": \"0.6547269754253718\", \"macro_user_top3\": \"0.841734839463901\", \"cluster_ci95_top1_low\": \"0.5965644564681961\", \"cluster_ci95_top1_high\": \"0.6865277802703998\", \"cluster_ci95_top3_low\": \"0.8044536926550897\", \"cluster_ci95_top3_high\": \"0.8595455960087095\", \"top1_seed_sd\": \"0.010989606710154688\", \"top3_seed_sd\": \"0.00042302000322679944\", \"macro_target_top1_seed_sd\": \"0.01894868669322078\", \"macro_target_top3_seed_sd\": \"0.004574994680479748\", \"macro_user_top1_seed_sd\": \"0.009704568969272733\", \"macro_user_top3_seed_sd\": \"0.0011327961284611167\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6569740607403305\", \"top3\": \"0.8356872519154436\", \"macro_target_top1\": \"0.5701158946043721\", \"macro_target_top3\": \"0.7122130350491778\", \"macro_user_top1\": \"0.6636911729202241\", \"macro_user_top3\": \"0.8418293052482178\", \"best_epoch\": \"29\", \"epochs_run\": \"30\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6350041539739685\", \"top3\": \"0.8354103203175482\", \"macro_target_top1\": \"0.5350037825268091\", \"macro_target_top3\": \"0.7050525817079413\", \"macro_user_top1\": \"0.6444215587805323\", \"macro_user_top3\": \"0.8405577684283118\", \"best_epoch\": \"24\", \"epochs_run\": \"29\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp2\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6465429705529401\", \"top3\": \"0.8362411151112342\", \"macro_target_top1\": \"0.5649094487328397\", \"macro_target_top3\": \"0.7135661297688842\", \"macro_user_top1\": \"0.6560681945753589\", \"macro_user_top3\": \"0.8428174447151737\", \"best_epoch\": \"28\", \"epochs_run\": \"30\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 2  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 2  Top-1: 64.62%  Top-3: 83.58%
Seeds 11, 23, 37 Top-3: [np.float64(83.57), np.float64(83.54), np.float64(83.62)]


---

## Imported section: `UNCP_3_Model_and_Results.ipynb`


# UNCP 3 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F
import math
import torch

def sinusoidal_positions(max_context, dimension):
    pe = torch.zeros(max_context, dimension)
    positions = torch.arange(max_context)[:, None]
    div = torch.exp(torch.arange(0, dimension, 2) * (-math.log(10000.0) / dimension))
    pe[:, 0::2] = torch.sin(positions * div)
    pe[:, 1::2] = torch.cos(positions * div)
    return pe

"""UNCP 3: GRU, Transformer, and a PyTorch SSD (Mamba-2-style) branch.

The SSD branch uses input-dependent scalar decay and B/C vectors per head.
It implements the recurrent form of structured state-space duality in plain
PyTorch, without claiming use of the optimized official CUDA Mamba2 kernel.
"""
import torch
from torch import nn
import torch.nn.functional as F



class SSDLayer(nn.Module):
    def __init__(self, dim, heads=4, state_size=16):
        super().__init__()
        assert dim % heads == 0
        self.heads, self.width, self.state_size = heads, dim // heads, state_size
        self.in_proj = nn.Linear(dim, dim + 2 * heads * state_size + heads + dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        batch, steps, dim = x.shape
        z = self.in_proj(x)
        w = self.heads * self.state_size
        u, b, c, decay, gate = torch.split(z, [dim, w, w, self.heads, dim], dim=-1)
        u = u.reshape(batch, steps, self.heads, self.width)
        b = b.reshape(batch, steps, self.heads, self.state_size)
        c = c.reshape(batch, steps, self.heads, self.state_size)
        decay = torch.sigmoid(decay)
        state = x.new_zeros(batch, self.heads, self.width, self.state_size)
        outputs = []
        for t in range(steps):
            state = decay[:, t, :, None, None] * state + u[:, t, :, :, None] * b[:, t, :, None, :]
            outputs.append((state * c[:, t, :, None, :]).sum(-1))
        y = torch.stack(outputs, dim=1).reshape(batch, steps, dim)
        return self.norm(x + self.out_proj(y * F.silu(gate)))


class UNCP3NextClick(nn.Module):
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config['embedding_dim'], config['hidden_dim']
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        layer = nn.TransformerEncoderLayer(dim, config['attention_heads'],
            config['transformer_feedforward'], config['dropout'], batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, config['transformer_layers'], enable_nested_tensor=False)
        self.ssd = SSDLayer(dim, config['attention_heads'])
        self.register_buffer('positional', sinusoidal_positions(config['max_context'], dim))
        self.transform_projection = nn.Linear(dim, hidden)
        self.ssd_projection = nn.Linear(dim, hidden)
        self.gate = nn.Linear(3 * hidden, 3)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config['dropout'])
        self.output = nn.Linear(hidden, n_classes)

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(len(x), device=x.device)
        embedded = self.embedding(x)
        gru, _ = self.gru(embedded)
        gru = gru[batch, lengths - 1]
        transformer = self.transformer(embedded + self.positional[:x.shape[1]], src_key_padding_mask=x.eq(0))
        transformer = self.transform_projection(transformer[batch, lengths - 1])
        ssd = self.ssd_projection(self.ssd(embedded)[batch, lengths - 1])
        states = torch.stack([gru, transformer, ssd], dim=1)
        weights = torch.softmax(self.gate(torch.cat([gru, transformer, ssd], dim=-1)), dim=-1)
        fused = self.norm((states * weights.unsqueeze(-1)).sum(dim=1))
        return self.output(self.dropout(fused))


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,attention_heads=4,transformer_feedforward=64,dropout=.2,transformer_layers=2,max_context=32)
model=UNCP3NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,6,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.isfinite(out).all()
print('UNCP 3 forward pass OK:',tuple(out.shape))

UNCP 3 forward pass OK: (2, 5)


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6112188067325147\", \"top3\": \"0.8139942767469768\", \"macro_target_top1\": \"0.5079396235824037\", \"macro_target_top3\": \"0.6855106314109176\", \"macro_user_top1\": \"0.6255331114932504\", \"macro_user_top3\": \"0.8202086292559252\", \"cluster_ci95_top1_low\": \"0.5595705902800073\", \"cluster_ci95_top1_high\": \"0.6622155941606191\", \"cluster_ci95_top3_low\": \"0.7800400898603109\", \"cluster_ci95_top3_high\": \"0.8387978780894642\", \"top1_seed_sd\": \"0.022853957728493367\", \"top3_seed_sd\": \"0.008237389744573671\", \"macro_target_top1_seed_sd\": \"0.03910456104403911\", \"macro_target_top3_seed_sd\": \"0.025677621025289022\", \"macro_user_top1_seed_sd\": \"0.025671229233503633\", \"macro_user_top3_seed_sd\": \"0.008054975730631787\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5947567617465153\", \"top3\": \"0.809009507984861\", \"macro_target_top1\": \"0.47728998319684945\", \"macro_target_top3\": \"0.6658837215105331\", \"macro_user_top1\": \"0.6036999829162267\", \"macro_user_top3\": \"0.812148212883942\", \"best_epoch\": \"19\", \"epochs_run\": \"24\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6373119172897628\", \"top3\": \"0.8235022616080495\", \"macro_target_top1\": \"0.5519805777876855\", \"macro_target_top3\": \"0.7145706062371812\", \"macro_user_top1\": \"0.6538133701687139\", \"macro_user_top3\": \"0.8282581532983488\", \"best_epoch\": \"30\", \"epochs_run\": \"30\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp3\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6015877411612665\", \"top3\": \"0.80947106064802\", \"macro_target_top1\": \"0.4945483097626758\", \"macro_target_top3\": \"0.6760775664850387\", \"macro_user_top1\": \"0.6190859813948106\", \"macro_user_top3\": \"0.8202195215854851\", \"best_epoch\": \"20\", \"epochs_run\": \"25\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 3  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 3  Top-1: 61.12%  Top-3: 81.40%
Seeds 11, 23, 37 Top-3: [np.float64(80.9), np.float64(82.35), np.float64(80.95)]


---

## Imported section: `UNCP_4_Model_and_Results.ipynb`


# UNCP 4 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

"""UNCP 4: typed graph attention, GRU, transition graph, adaptive fusion.

The candidate mask is a conservative observed-page proxy. It is built only
from a supplied UI inventory and training-fold transitions, never target pages.
"""
import torch
from torch import nn
import torch.nn.functional as F


class TypedGraphAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.query = nn.ModuleList([nn.Linear(dim, dim) for _ in range(2)])
        self.key = nn.ModuleList([nn.Linear(dim, dim) for _ in range(2)])
        self.value = nn.ModuleList([nn.Linear(dim, dim) for _ in range(2)])
        self.fusion = nn.Linear(3 * dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, nodes, relations):
        channels = [nodes]
        for i, edges in enumerate(relations):
            scores = self.query[i](nodes) @ self.key[i](nodes).T / nodes.shape[-1] ** .5
            weights = scores.masked_fill(~edges, -1e9).softmax(-1)
            channels.append(weights @ self.value[i](nodes))
        return self.norm(nodes + F.gelu(self.fusion(torch.cat(channels, -1))))


class UNCP4NextClick(nn.Module):
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config['embedding_dim'], config['hidden_dim']
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.page_embedding = nn.Embedding(config['n_pages'], dim)
        self.graph = TypedGraphAttention(dim)
        self.gru = nn.GRU(dim, hidden, batch_first=True)
        self.graph_projection = nn.Linear(dim, hidden)
        self.transition_projection = nn.Linear(n_classes, hidden)
        self.gate = nn.Linear(3 * hidden, 3)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config['dropout'])
        self.output = nn.Linear(hidden, n_classes)
        self.register_buffer('page_edges', config['page_edges'].bool())
        self.register_buffer('transition_edges', config['transition_edges'].bool())
        self.register_buffer('transition_probs', config['transition_probs'].float())
        self.register_buffer('candidate_mask', config['candidate_mask'].bool())

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch = torch.arange(x.shape[0], device=x.device)
        nodes = torch.cat([self.embedding.weight, self.page_embedding.weight], 0)
        graph_nodes = self.graph(nodes, (self.page_edges, self.transition_edges))
        graph_tokens = graph_nodes[x]
        recurrent, _ = self.gru(self.embedding(x) + graph_tokens)
        recurrent = recurrent[batch, lengths - 1]
        observed = x.ne(0).float().unsqueeze(-1)
        graph_state = self.graph_projection((graph_tokens * observed).sum(1) / observed.sum(1).clamp_min(1))
        last = x[batch, lengths - 1]
        transition_state = self.transition_projection(self.transition_probs[last])
        states = torch.stack([recurrent, graph_state, transition_state], 1)
        weights = torch.softmax(self.gate(torch.cat([recurrent, graph_state, transition_state], -1)), -1)
        logits = self.output(self.dropout(self.norm((states * weights.unsqueeze(-1)).sum(1))))
        return logits.masked_fill(~self.candidate_mask[last], -1e9)


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,dropout=.2,n_pages=2,page_edges=torch.eye(12,dtype=torch.bool),transition_edges=torch.eye(12,dtype=torch.bool),transition_probs=torch.full((10,5),.2),candidate_mask=torch.ones(10,5,dtype=torch.bool))
model=UNCP4NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,6,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.isfinite(out).all()
print('UNCP 4 forward pass OK:',tuple(out.shape))

UNCP 4 forward pass OK: (2, 5)


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5984491830517862\", \"top3\": \"0.8107941782824087\", \"macro_target_top1\": \"0.5156944881594836\", \"macro_target_top3\": \"0.6801169307784917\", \"macro_user_top1\": \"0.6107045989967255\", \"macro_user_top3\": \"0.818546311470632\", \"cluster_ci95_top1_low\": \"0.5487152094633457\", \"cluster_ci95_top1_high\": \"0.6477397189779494\", \"cluster_ci95_top3_low\": \"0.7803692616815592\", \"cluster_ci95_top3_high\": \"0.8358363308189471\", \"top1_seed_sd\": \"0.013969118388647223\", \"top3_seed_sd\": \"0.008787135234241824\", \"macro_target_top1_seed_sd\": \"0.017129077511917827\", \"macro_target_top3_seed_sd\": \"0.01779798043601871\", \"macro_user_top1_seed_sd\": \"0.013848603369107029\", \"macro_user_top3_seed_sd\": \"0.00667036035667742\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5938336564201976\", \"top3\": \"0.8091941290501247\", \"macro_target_top1\": \"0.5118135455238988\", \"macro_target_top3\": \"0.6665667087872567\", \"macro_user_top1\": \"0.5998210975130807\", \"macro_user_top3\": \"0.8167574876752866\", \"best_epoch\": \"21\", \"epochs_run\": \"26\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.5873719191359734\", \"top3\": \"0.802917012831164\", \"macro_target_top1\": \"0.5008388584534144\", \"macro_target_top3\": \"0.6735106472058255\", \"macro_user_top1\": \"0.6060001257184275\", \"macro_user_top3\": \"0.8129527506988308\", \"best_epoch\": \"19\", \"epochs_run\": \"24\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp4\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6141419735991877\", \"top3\": \"0.8202713929659374\", \"macro_target_top1\": \"0.5344310605011375\", \"macro_target_top3\": \"0.7002734363423929\", \"macro_user_top1\": \"0.6262925737586683\", \"macro_user_top3\": \"0.8259286960377783\", \"best_epoch\": \"29\", \"epochs_run\": \"30\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 4  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 4  Top-1: 59.84%  Top-3: 81.08%
Seeds 11, 23, 37 Top-3: [np.float64(80.92), np.float64(80.29), np.float64(82.03)]


---

## Imported section: `UNCP_5_Model_and_Results.ipynb`


# UNCP 5 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
"""UNCP 5: validation-selected fusion of UNCP 2 and a training-only transition model.

Run after the three UNCP 2 grouped-user checkpoints exist. The test set is
accessed only after the fusion rule has been selected on validation data.
"""
import collections
import csv
import json
import math
from pathlib import Path

import numpy as np
import torch
# build_model is required only by main() during full training
# Full training needs the benchmark helpers from the accompanying project

HERE = Path.cwd()
SOURCE = HERE / 'results_uncp2'
OUT = HERE / 'results_adaptive_fusion'
SEEDS = (11, 23, 37)


def transition_tables(train, encoder):
    tables = [collections.defaultdict(collections.Counter) for _ in range(4)]
    global_counts = np.ones(len(encoder.outputs), dtype=np.float64) * .1
    for item in train:
        target = encoder.outputs[item['target']]
        global_counts[target] += 1
        context = item['context']
        for order in range(1, min(4, len(context)) + 1):
            tables[order-1][tuple(context[-order:])][target] += 1
    return tables, global_counts / global_counts.sum()


def transition_predictions(data, tables, prior):
    probs = np.empty((len(data), len(prior)), dtype=np.float64)
    supports = np.zeros(len(data), dtype=np.float64)
    for i, item in enumerate(data):
        context = item['context']
        counter = None
        for order in range(min(4, len(context)), 0, -1):
            found = tables[order-1].get(tuple(context[-order:]))
            if found:
                counter = found
                break
        if counter is None:
            probs[i] = prior
        else:
            support = sum(counter.values())
            supports[i] = support
            p = prior * 2.0
            for target, count in counter.items():
                p[target] += count
            probs[i] = p / p.sum()
    return probs, supports


def correct(probs, labels):
    ranked = np.argsort(-probs, axis=1, kind='stable')[:, :3]
    return np.column_stack((ranked[:, 0] == labels, (ranked == labels[:, None]).any(1))).astype(float)


def blend(neural, transitions, support, params):
    intercept, slope, entropy_slope = params
    ent_neural = -(neural * np.log(np.maximum(neural, 1e-12))).sum(1)
    ent_trans = -(transitions * np.log(np.maximum(transitions, 1e-12))).sum(1)
    raw = np.clip(intercept + slope * np.log1p(support) + entropy_slope * (ent_neural-ent_trans), -30, 30)
    transition_weight = 1 / (1 + np.exp(-raw))
    return (1-transition_weight[:, None]) * neural + transition_weight[:, None] * transitions, transition_weight


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    torch.set_num_threads(2)
    records = json.loads((SOURCE/'records_restricted.json').read_text())
    parts, manifest = split_records(records)
    group = parts['grouped_user']
    datasets = {part: examples(rows) for part, rows in group.items()}
    encoder = Encoder(datasets['train'])
    encoded = {part: encoder.transform(rows) for part, rows in datasets.items()}
    tables, prior = transition_tables(datasets['train'], encoder)
    trans_val, support_val = transition_predictions(datasets['val'], tables, prior)
    labels_val = encoded['val'][2].numpy()
    val_neural = []
    config = dict(CONFIG, graph_adj=graph_adjacency(encoded['train'], encoder))
    for seed in SEEDS:
        checkpoint = torch.load(SOURCE/f'grouped_user_uncp2_seed{seed}.pt', map_location='cpu', weights_only=False)
        assert checkpoint['input_vocabulary'] == encoder.inputs
        assert checkpoint['output_vocabulary'] == encoder.outputs
        model = build_model('uncp2', len(encoder.inputs)+2, len(encoder.outputs), config)
        model.load_state_dict(checkpoint['state_dict'])
        val_neural.append(predict(model, encoded['val']).astype(np.float64))
    # Small, fixed search space. Select by mean validation Top-3, then Top-1,
    # then lower NLL; the held-out test data have not been read at this point.
    choices = [(a,b,c) for a in (-4,-2,0,2,4)
               for b in (-1,-.5,0,.5,1)
               for c in (-1,0,1)]
    choices.append((-30,0,0))  # essentially pure UNCP 2
    scored = []
    for params in choices:
        values = []
        for neural in val_neural:
            mixed,_ = blend(neural, trans_val, support_val, params)
            hits = correct(mixed, labels_val)
            known = labels_val >= 0
            nll = -np.log(np.maximum(mixed[np.flatnonzero(known), labels_val[known]], 1e-12)).mean()
            values.append((hits[:,1].mean(), hits[:,0].mean(), nll))
        mean = np.mean(values, axis=0)
        scored.append((mean[0], mean[1], -mean[2], params))
    chosen = max(scored)
    params = chosen[3]
    validation_baseline_top3 = float(np.mean([correct(p,labels_val)[:,1].mean() for p in val_neural]))
    # The test set enters only after the validation choice is frozen.
    trans_test, support_test = transition_predictions(datasets['test'], tables, prior)
    labels_test = encoded['test'][2].numpy()
    runs = []
    fusion_hits = []
    baseline_hits = []
    for seed in SEEDS:
        saved = np.load(SOURCE/f'grouped_user_uncp2_seed{seed}_predictions_restricted.npz')
        assert np.array_equal(saved['targets'], labels_test)
        neural = saved['probabilities'].astype(np.float64)
        fused, weights = blend(neural, trans_test, support_test, params)
        hits = correct(fused, labels_test)
        baseline = correct(neural, labels_test)
        runs.append({'seed':seed, 'top1':float(hits[:,0].mean()),'top3':float(hits[:,1].mean()),
                     'uncp2_top1':float(baseline[:,0].mean()),'uncp2_top3':float(baseline[:,1].mean()),
                     'mean_transition_weight':float(weights.mean())})
        fusion_hits.append(hits)
        baseline_hits.append(baseline)
        np.savez_compressed(OUT/f'grouped_user_adaptive_seed{seed}.npz', correct=hits, targets=labels_test,
                            probabilities=fused.astype(np.float32))
    with (OUT/'grouped_user_adaptive_runs.csv').open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(runs[0]));writer.writeheader();writer.writerows(runs)
    difference = np.mean(fusion_hits,axis=0) - np.mean(baseline_hits,axis=0)
    user_indices=collections.defaultdict(list)
    for index,item in enumerate(datasets['test']):
        user_indices[item['user']].append(index)
    user_sums=np.array([difference[idx].sum(0) for idx in user_indices.values()])
    user_counts=np.array([len(idx) for idx in user_indices.values()])
    draws=np.random.default_rng(20261104).integers(0,len(user_indices),(5000,len(user_indices)))
    boot=user_sums[draws].sum(1)/user_counts[draws].sum(1)[:,None]
    interval=np.quantile(boot,[.025,.975],axis=0)
    frequency_rows=[]
    for group,low,high in [('unseen_0',0,0),('rare_1_19',1,19),('medium_20_99',20,99),('frequent_100plus',100,math.inf)]:
        idx=[i for i,item in enumerate(datasets['test']) if low<=encoder.target_counts[item['target']]<=high]
        if idx:
            frequency_rows.append({'group':group,'n':len(idx),
                'uncp2_top3':float(np.mean(baseline_hits,axis=0)[idx,1].mean()),
                'fusion_top3':float(np.mean(fusion_hits,axis=0)[idx,1].mean())})
    with (OUT/'grouped_user_frequency_comparison.csv').open('w',newline='') as stream:
        writer=csv.DictWriter(stream,fieldnames=list(frequency_rows[0]));writer.writeheader();writer.writerows(frequency_rows)
    report={'model':'UNCP 5','architecture':'validation-selected UNCP 2 and training-only transition fusion','protocol':'grouped_user',
            'seeds':SEEDS,'test_examples':len(labels_test),'test_users':len(manifest['grouped_user']['test']['user_ids']),
            'selected_params':dict(intercept=params[0],log_support_slope=params[1],entropy_difference_slope=params[2]),
            'validation_mean_top3':float(chosen[0]),'validation_mean_top1':float(chosen[1]),
            'validation_uncp2_mean_top3':validation_baseline_top3,
            'test_mean_top1':float(np.mean([r['top1'] for r in runs])),
            'test_mean_top3':float(np.mean([r['top3'] for r in runs])),
            'test_top1_seed_sd':float(np.std([r['top1'] for r in runs],ddof=1)),
            'test_top3_seed_sd':float(np.std([r['top3'] for r in runs],ddof=1)),
            'uncp2_mean_top1':float(np.mean([r['uncp2_top1'] for r in runs])),
            'uncp2_mean_top3':float(np.mean([r['uncp2_top3'] for r in runs])),
            'mean_transition_weight':float(np.mean([r['mean_transition_weight'] for r in runs])),
            'paired_top1_difference':float(difference[:,0].mean()),
            'paired_top3_difference':float(difference[:,1].mean()),
            'paired_user_cluster_ci95_top3':[float(interval[0,1]),float(interval[1,1])],
            'selection':'fixed 76-rule grid, validation mean Top-3 then Top-1 then NLL; test accessed after selection'}
    (OUT/'adaptive_fusion_report.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(report,indent=2))




In [2]:
neural=np.array([[.15,.55,.30],[.60,.20,.20]])
transition=np.array([[.10,.70,.20],[.50,.40,.10]])
combined,weights=blend(neural,transition,np.array([5.,20.]),(2,-1,1))
assert combined.shape==(2,3) and np.allclose(combined.sum(1),1)
print('UNCP 5 adaptive fusion OK:',np.round(weights,4).tolist())

UNCP 5 adaptive fusion OK: [0.5941, 0.2616]


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"model\": \"UNCP 5\", \"architecture\": \"validation-selected UNCP 2 and training-only transition fusion\", \"protocol\": \"grouped_user\", \"seeds\": [11, 23, 37], \"test_examples\": 10833, \"test_users\": 11, \"selected_params\": {\"intercept\": 2, \"log_support_slope\": -1, \"entropy_difference_slope\": 1}, \"validation_mean_top3\": 0.8345820990881107, \"validation_mean_top1\": 0.6578962653191613, \"validation_uncp2_mean_top3\": 0.825056630074926, \"test_mean_top1\": 0.6670974491522816, \"test_mean_top3\": 0.8508569494445982, \"test_top1_seed_sd\": 0.008422714353512705, \"test_top3_seed_sd\": 0.002027332091452598, \"uncp2_mean_top1\": 0.646173728422413, \"uncp2_mean_top3\": 0.8357795624480753, \"mean_transition_weight\": 0.21119532680331224, \"paired_top1_difference\": 0.020923720729868613, \"paired_top3_difference\": 0.01507738699652297, \"paired_user_cluster_ci95_top3\": [0.010282369312566746, 0.017732589850673244], \"selection\": \"fixed 76-rule grid, validation mean Top-3 then Top-1 then NLL; test accessed after selection\"}, \"runs\": [{\"seed\": \"11\", \"top1\": \"0.6746053724729991\", \"top3\": \"0.8521185267238992\", \"uncp2_top1\": \"0.6569740607403305\", \"uncp2_top3\": \"0.8356872519154436\", \"mean_transition_weight\": \"0.20852521717631758\"}, {\"seed\": \"23\", \"top1\": \"0.65798947659928\", \"top3\": \"0.84851841595126\", \"uncp2_top1\": \"0.6350041539739685\", \"uncp2_top3\": \"0.8354103203175482\", \"mean_transition_weight\": \"0.21616524434512732\"}, {\"seed\": \"37\", \"top1\": \"0.6686974983845657\", \"top3\": \"0.8519339056586357\", \"uncp2_top1\": \"0.6465429705529401\", \"uncp2_top3\": \"0.8362411151112342\", \"mean_transition_weight\": \"0.20889551888849184\"}], \"behavior\": null}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 5  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 5  Top-1: 66.71%  Top-3: 85.09%
Seeds 11, 23, 37 Top-3: [np.float64(85.21), np.float64(84.85), np.float64(85.19)]


---

## Imported section: `UNCP_6_Model_and_Results.ipynb`


# UNCP 6 | Model and executed results

کد مدل و یک بررسی اجرایی در همین دفترچه اجرا شده‌اند. اعداد سه seed از فایل‌های نتیجهٔ آزمایش کامل که پیش‌تر اجرا شده‌اند، در سلول آخر ثبت و با میانگین همان فایل‌ها کنترل می‌شوند. این دفترچه آموزش کامل سه seed را دوباره اجرا نمی‌کند؛ برای بازآموزی، داده، کد benchmark و تنظیمات همان پروژه لازم است.

In [1]:
import torch
from torch import nn
import torch.nn.functional as F

"""UNCP 6: repeat-aware dual-scale click model.

One GRU reads every observed click; a second GRU reads consecutive runs with
their lengths. A repeat-vs-switch hazard and a conditional switch classifier
form a normalized next-click distribution. No target information is input.
"""
import torch
from torch import nn
import torch.nn.functional as F


class UNCP6NextClick(nn.Module):
    def __init__(self, vocab_size, n_classes, config):
        super().__init__()
        dim, hidden = config['embedding_dim'], config['hidden_dim']
        self.embedding = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.run_length_embedding = nn.Embedding(config['max_context'] + 1, dim)
        self.raw_gru = nn.GRU(dim, hidden, batch_first=True)
        self.run_gru = nn.GRU(dim, hidden, batch_first=True)
        self.gate = nn.Linear(2 * hidden, hidden)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(config['dropout'])
        self.repeat_head = nn.Linear(hidden, 1)
        self.switch_head = nn.Linear(hidden, n_classes)
        self.register_buffer('repeat_class', config['repeat_class'].long())

    def forward(self, x, lengths):
        x = x[:, :int(lengths.max())]
        batch, steps = x.shape
        rows = torch.arange(batch, device=x.device)
        positions = torch.arange(steps, device=x.device)[None, :]
        valid = positions < lengths[:, None]
        keep = valid.clone()
        keep[:, 1:] &= x[:, 1:].ne(x[:, :-1])
        destination = keep.long().cumsum(1) - 1
        compact = torch.zeros_like(x)
        compact[rows[:, None].expand_as(x)[keep], destination[keep]] = x[keep]
        run_lengths = torch.zeros_like(x)
        run_lengths.scatter_add_(1, destination.clamp_min(0), valid.long())
        compact_lengths = keep.sum(1)
        raw, _ = self.raw_gru(self.embedding(x))
        raw = raw[rows, lengths - 1]
        run_inputs = self.embedding(compact) + self.run_length_embedding(run_lengths.clamp_max(self.run_length_embedding.num_embeddings - 1))
        runs, _ = self.run_gru(run_inputs)
        runs = runs[rows, compact_lengths - 1]
        gate = torch.sigmoid(self.gate(torch.cat([raw, runs], dim=-1)))
        state = self.dropout(self.norm(gate * raw + (1 - gate) * runs))
        repeat_logit = self.repeat_head(state).squeeze(-1)
        switch_logits = self.switch_head(state)
        last = x[rows, lengths - 1]
        repeat_index = self.repeat_class[last]
        has_repeat_class = repeat_index.ge(0)
        active_repeat_index = repeat_index.clamp_min(0)
        mask = F.one_hot(active_repeat_index, switch_logits.shape[-1]).bool() & has_repeat_class[:, None]
        switch_logits = switch_logits.masked_fill(mask, -1e9)
        switch_log_probs = F.log_softmax(switch_logits, dim=-1)
        repeat_log_prob = F.logsigmoid(repeat_logit)
        switch_log_prob = F.logsigmoid(-repeat_logit)
        # Unknown input labels cannot be repeated in the training target
        # vocabulary, so the switch distribution receives full mass.
        switch_log_prob = torch.where(has_repeat_class, switch_log_prob, torch.zeros_like(switch_log_prob))
        result = switch_log_probs + switch_log_prob[:, None]
        return torch.where(mask, repeat_log_prob[:, None], result)


In [2]:
cfg=dict(embedding_dim=32,hidden_dim=64,max_context=32,dropout=.2,repeat_class=torch.tensor([-1,-1,0,1,2,3,4,0,1,2]))
model=UNCP6NextClick(10,5,cfg)
x=torch.tensor([[2,2,3,0],[4,5,5,7]])
lengths=torch.tensor([3,4])
out=model(x,lengths)
assert out.shape==(2,5) and torch.allclose(out.exp().sum(1),torch.ones(2),atol=1e-5)
print('UNCP 6 forward pass OK:',tuple(out.shape),'normalized probabilities')

UNCP 6 forward pass OK: (2, 5) normalized probabilities


## Recorded experiment results

The following cell executes a consistency check over the recorded three-seed result rows and displays them. It does not retrain the model.

In [3]:
import json, numpy as np
record = json.loads("{\"summary\": {\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seeds\": \"3\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6659897227607002\", \"top3\": \"0.840825871565279\", \"macro_target_top1\": \"0.5702779086941139\", \"macro_target_top3\": \"0.7281072101547532\", \"macro_user_top1\": \"0.6656282403262328\", \"macro_user_top3\": \"0.8418568215426565\", \"cluster_ci95_top1_low\": \"0.6091920450994863\", \"cluster_ci95_top1_high\": \"0.6995171213041147\", \"cluster_ci95_top3_low\": \"0.802033844632611\", \"cluster_ci95_top3_high\": \"0.8642404745892035\", \"top1_seed_sd\": \"0.005097458602388681\", \"top3_seed_sd\": \"0.0021635614426593584\", \"macro_target_top1_seed_sd\": \"0.011683224489405347\", \"macro_target_top3_seed_sd\": \"0.011689404952768491\", \"macro_user_top1_seed_sd\": \"0.005838459964558864\", \"macro_user_top3_seed_sd\": \"0.002589649228566629\"}, \"runs\": [{\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seed\": \"11\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6703590879719376\", \"top3\": \"0.838364257361765\", \"macro_target_top1\": \"0.5756091800875947\", \"macro_target_top3\": \"0.7365269857757945\", \"macro_user_top1\": \"0.6678463424106305\", \"macro_user_top3\": \"0.8396346576501166\", \"best_epoch\": \"24\", \"epochs_run\": \"29\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seed\": \"23\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6603895504477061\", \"top3\": \"0.842425920797563\", \"macro_target_top1\": \"0.5568800368153548\", \"macro_target_top3\": \"0.7147609816096553\", \"macro_user_top1\": \"0.6590057845179073\", \"macro_user_top3\": \"0.8447007548538839\", \"best_epoch\": \"20\", \"epochs_run\": \"25\"}, {\"protocol\": \"grouped_user\", \"model\": \"uncp6\", \"seed\": \"37\", \"test_examples\": \"10833\", \"test_users\": \"11\", \"top1\": \"0.6672205298624573\", \"top3\": \"0.8416874365365088\", \"macro_target_top1\": \"0.5783445091793922\", \"macro_target_top3\": \"0.7330336630788102\", \"macro_user_top1\": \"0.6700325940501609\", \"macro_user_top3\": \"0.8412350521239692\", \"best_epoch\": \"25\", \"epochs_run\": \"30\"}], \"behavior\": [{\"group\": \"all\", \"n\": 10833, \"uncp6_top1\": 0.6659897227607003, \"uncp6_top3\": 0.8408258715652789, \"uncp5_top1\": 0.6670974491522816, \"uncp5_top3\": 0.8508569494445982}, {\"group\": \"repeat\", \"n\": 1541, \"uncp6_top1\": 0.8407960199004977, \"uncp6_top3\": 0.9409474367293965, \"uncp5_top1\": 0.8654553320354749, \"uncp5_top3\": 0.9407311269738265}, {\"group\": \"switch\", \"n\": 9292, \"uncp6_top1\": 0.6369995695221696, \"uncp6_top3\": 0.8242215525900415, \"uncp5_top1\": 0.6342014636246233, \"uncp5_top3\": 0.8359520734682163}]}")
summary, runs = record['summary'], record['runs']
top1 = float(summary.get('test_mean_top1', summary.get('top1')))
top3 = float(summary.get('test_mean_top3', summary.get('top3')))
seed_top1 = np.array([float(row['top1']) for row in runs])
seed_top3 = np.array([float(row['top3']) for row in runs])
assert np.isclose(seed_top1.mean(), top1, atol=1e-9)
assert np.isclose(seed_top3.mean(), top3, atol=1e-9)
print('Recorded grouped-user experiment: 10,833 test targets, 11 held-out users')
print(f'UNCP 6  Top-1: {100*top1:.2f}%  Top-3: {100*top3:.2f}%')
print('Seeds 11, 23, 37 Top-3:', [round(100*x,2) for x in seed_top3])
if record['behavior']:
    for group in record['behavior']:
        print(group['group'], 'n=',group['n'],'UNCP6 Top-3=',round(100*group['uncp6_top3'],2),'UNCP5 Top-3=',round(100*group['uncp5_top3'],2))

Recorded grouped-user experiment: 10,833 test targets, 11 held-out users
UNCP 6  Top-1: 66.60%  Top-3: 84.08%
Seeds 11, 23, 37 Top-3: [np.float64(83.84), np.float64(84.24), np.float64(84.17)]
all n= 10833 UNCP6 Top-3= 84.08 UNCP5 Top-3= 85.09
repeat n= 1541 UNCP6 Top-3= 94.09 UNCP5 Top-3= 94.07
switch n= 9292 UNCP6 Top-3= 82.42 UNCP5 Top-3= 83.6
